# Notebook Overview — Validate and Tune Two Models

## Purpose

This notebook performs **final validation and comparison** of the selected classifiers for the Digital Image Processing (DIP)–based AI image detection pipeline.

It evaluates the retained models (**RBF SVM** and **MLP**) using **stratified cross-validation** on the normalized training dataset, producing a consistent and unbiased comparison prior to final independent test evaluation.

---

## Inputs

Required inputs:

* Normalized training feature vector CSV:
  * `metadata/vectors/<train_feature_vectors_normalized_filename>`

Shared configuration:

* `src/project_config.py`

---

## Execution Model

* Normalized training dataset is loaded
* Dataset structure, metadata, and feature dimensionality are validated
* Feature matrix (`X_train`) and encoded labels (`y_train`) are prepared
* Final classifier configurations (RBF SVM and MLP) are defined
* Stratified k-fold cross-validation is applied to both classifiers
* Performance metrics are computed:
  * Accuracy
  * Precision
  * Recall
  * F1 score
  * ROC AUC
* Results are aggregated (mean and standard deviation)
* Classifiers are ranked based on ROC AUC
* A condensed comparison table is generated
* Validation outputs are saved to CSV files
* Saved outputs are reloaded and verified for consistency
* Processing is deterministic and reproducible

---

## Outputs

**Cross-Validation Results**  
`metadata/results/<cross_validation_tuned_results_filename>`

**Tuned Classifier Comparison**  
`metadata/results/<classifier_comparison_tuned_filename>`

Each output includes:

* Classifier name
* Mean and standard deviation of evaluation metrics
* Condensed comparison of key performance metrics

---

## Expected Behavior

* Training dataset passes all validation checks
* Feature dimensionality remains consistent (26 features)
* No missing values are present in the dataset
* Cross-validation executes successfully across all folds
* Both classifiers (RBF SVM and MLP) are evaluated consistently
* Performance metrics are stable and comparable across models
* Results are correctly ranked by ROC AUC
* Output files are saved and verified without error
* Outputs are ready for final independent test evaluation

---
---

### 🔷 Step 1 — Startup: Environment, Paths, and Verification

- Import required libraries for classifier validation and analysis
- Configure notebook display behavior using `VERBOSE`
- Optionally suppress warnings for cleaner output
- Clone the GitHub repository if needed
- Add the project `src` directory to the Python path
- Import shared configuration values from `project_config.py`
- Define input path for normalized training feature vectors
- Define paths for cross-validation results and tuned classifier comparison
- Create results output directory if it does not exist
- Verify that required input data is available
- Optionally display configuration paths when `VERBOSE=True`

---

In [1]:
# ============================================
# Step 1: Startup — Environment, Paths, and Verification
# ============================================

# -------------------------------------------------
# Import required libraries
# -------------------------------------------------
import os
import sys
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Suppress warnings for cleaner output (optional)
# -------------------------------------------------
if not VERBOSE:
    warnings.filterwarnings("ignore")

# -------------------------------------------------
# Clone GitHub repository if needed
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Add src directory to Python path
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_NORMALIZED_PATH,
    NUM_FEATURES,
    RANDOM_SEED,
    METADATA_COLUMNS,
    AI_LABEL,
    REAL_LABEL,
    K_FOLDS,
    CV_SHUFFLE,
    CV_RANDOM_STATE,
    RESULTS_METADATA_DIR,
    CROSS_VALIDATION_TUNED_RESULTS_PATH,
    CLASSIFIER_COMPARISON_TUNED_PATH,
)

# -------------------------------------------------
# Convert configured paths to Path objects
# -------------------------------------------------
TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = Path(TRAIN_NORMALIZED_PATH)

RESULTS_DIR = Path(RESULTS_METADATA_DIR)

CV_RESULTS_CSV_PATH = Path(CROSS_VALIDATION_TUNED_RESULTS_PATH)
TUNED_COMPARISON_CSV_PATH = Path(CLASSIFIER_COMPARISON_TUNED_PATH)

# -------------------------------------------------
# Ensure required output directory exists
# -------------------------------------------------
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required input files...\n")

missing_files = []

if not TRAIN_FEATURE_VECTORS_NORMALIZED_CSV.exists():
    missing_files.append(str(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV))

if missing_files:
    raise FileNotFoundError(
        "Missing required input file(s):\n" + "\n".join(missing_files)
    )

print("All required input files are present.")

# -------------------------------------------------
# Optional verbose display of configuration paths
# -------------------------------------------------
if VERBOSE:
    print(f"Training data       : {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")
    print(f"Results directory   : {RESULTS_DIR}")
    print(f"CV results CSV      : {CV_RESULTS_CSV_PATH}")
    print(f"Tuned comparison CSV: {TUNED_COMPARISON_CSV_PATH}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nLibraries imported successfully.")



Cloning repository...
Verifying required input files...

All required input files are present.
Training data       : /content/dip-ai-image-detection/metadata/vectors/train_feature_vectors_normalized.csv
Results directory   : /content/dip-ai-image-detection/metadata/results
CV results CSV      : /content/dip-ai-image-detection/metadata/results/cross_validation_tuned_results.csv
Tuned comparison CSV: /content/dip-ai-image-detection/metadata/results/classifier_comparison_tuned.csv

Libraries imported successfully.


### 🔷 Step 2 — Load Normalized Training Data

- Load normalized training feature vector dataset from CSV
- Confirm successful data loading
- Optionally display dataset shape and structure when `VERBOSE=True`
- List column names for inspection
- Preview first few rows of the dataset
- Prepare dataset for validation and cross-validation steps

---

In [2]:
# ============================================
# Step 2: Load Normalized Training Data
# ============================================

print("Loading normalized training dataset...\n")

# -------------------------------------------------
# Load normalized training feature vector CSV
# -------------------------------------------------
df_train = pd.read_csv(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV)

print("Training dataset loaded successfully.")

# -------------------------------------------------
# Optional verbose display of dataset information
# -------------------------------------------------
if VERBOSE:
    print(f"\nShape: {df_train.shape}")

    print("\nColumn names:")
    for col in df_train.columns:
        print(col)

    print("\nFirst 5 rows:")
    try:
        display(df_train.head())
    except:
        print(df_train.head())



Loading normalized training dataset...

Training dataset loaded successfully.

Shape: (14400, 30)

Column names:
filename
class_label
source_dataset
subset
Mean Gradient
Std Gradient
Max Gradient
Gradient Entropy
Edge Density
Orientation Mean
Orientation Std
Orientation Entropy
Global Entropy
Local Entropy Mean
Local Entropy Std
Intensity Mean
Intensity Std
Laplacian Variance
Patch Variance Mean
Patch Variance Std
Noise Residual Energy
Low Frequency Energy Ratio
Mid Frequency Energy Ratio
High Frequency Energy Ratio
Radial Mean
Radial Std
Radial Entropy
Spectral Centroid
Spectral Bandwidth
Log Spectrum Std

First 5 rows:


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Noise Residual Energy,Low Frequency Energy Ratio,Mid Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,0.973214,1.208264,0.913526,0.776979,0.735101,-0.207859,...,1.071297,-0.189219,0.111086,0.417247,0.448644,0.439427,-0.549996,-0.084929,0.290945,-1.254194
1,rl_coco_001397.png,rl,MS_COCO_2017,train,-0.450913,-0.225225,0.641767,-0.488332,-0.447341,-0.246587,...,-0.330229,-0.264290,0.292101,0.144170,-1.327122,-1.325683,1.397134,0.856623,0.808074,0.071118
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,0.605933,0.723094,0.717432,0.610900,0.160581,-0.468100,...,0.851489,-0.544033,0.442499,0.805334,-0.343146,-0.332461,-0.549996,-0.217414,0.432281,-1.341491
3,rl_coco_000800.png,rl,MS_COCO_2017,train,0.021388,0.357107,0.528482,-0.077406,0.331237,1.075664,...,0.188882,0.518813,-0.552523,-0.349898,1.963337,1.956027,-0.549996,-0.448685,-0.628062,-1.111831
4,ai_mj_002892.png,ai,Midjourney,train,-0.212459,-0.661686,-0.209270,0.288772,-0.061089,-0.293840,...,-0.281104,0.607047,-0.600533,-0.556607,-0.011009,-0.013482,-0.549996,-0.185376,-0.470299,0.325939


### 🔷 Step 3 — Validate Training Data and Prepare Classifier Inputs

- Verify presence of required metadata columns
- Identify feature columns and validate feature count
- Ensure no missing values are present in metadata or feature columns
- Validate class labels against expected values (`ai`, `real`)
- Construct feature matrix `X_train` and label vector `y_labels`
- Encode class labels into numeric format using `LabelEncoder`
- Optionally display dataset shapes and label mappings when `VERBOSE=True`
- Confirm dataset is valid and ready for cross-validation

---

In [3]:
# ============================================
# Step 3: Validate Training Data and Prepare Classifier Inputs
# ============================================

print("Validating training dataset and preparing classifier inputs...\n")

# -------------------------------------------------
# Verify required metadata columns are present
# -------------------------------------------------
missing_metadata_cols = [col for col in METADATA_COLUMNS if col not in df_train.columns]

if missing_metadata_cols:
    raise ValueError(f"Missing required metadata columns: {missing_metadata_cols}")

if VERBOSE:
    print("Metadata columns verified.")

# -------------------------------------------------
# Identify and validate feature columns
# -------------------------------------------------
feature_columns = [col for col in df_train.columns if col not in METADATA_COLUMNS]

if VERBOSE:
    print(f"Number of feature columns found: {len(feature_columns)}")

if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} feature columns, but found {len(feature_columns)}."
    )

if VERBOSE:
    print("Feature count verified.")

# -------------------------------------------------
# Check for missing values across dataset
# -------------------------------------------------
if df_train[METADATA_COLUMNS + feature_columns].isnull().any().any():
    null_counts = df_train[METADATA_COLUMNS + feature_columns].isnull().sum()
    null_counts = null_counts[null_counts > 0]
    raise ValueError(f"Missing values detected:\n{null_counts}")

if VERBOSE:
    print("No missing values detected.")

# -------------------------------------------------
# Validate class label values
# -------------------------------------------------
unique_labels = sorted(df_train["class_label"].unique().tolist())
expected_labels = sorted([AI_LABEL, REAL_LABEL])

if VERBOSE:
    print(f"Observed class labels: {unique_labels}")

if unique_labels != expected_labels:
    raise ValueError(
        f"Expected class labels {expected_labels}, but found {unique_labels}."
    )

if VERBOSE:
    print("Class labels verified.")

# -------------------------------------------------
# Prepare feature matrix (X) and target vector (y)
# -------------------------------------------------
X_train = df_train[feature_columns].to_numpy()
y_labels = df_train["class_label"].to_numpy()

# -------------------------------------------------
# Encode class labels to numeric format
# -------------------------------------------------
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_labels)

print("Classifier input arrays prepared successfully.")

# -------------------------------------------------
# Optional verbose display of dataset properties
# -------------------------------------------------
if VERBOSE:
    print(f"\nX_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")

    label_mapping = {
        class_name: int(label_encoder.transform([class_name])[0])
        for class_name in label_encoder.classes_
    }

    print("\nEncoded label mapping:")
    for class_name, encoded_value in label_mapping.items():
        print(f"  {class_name} -> {encoded_value}")



Validating training dataset and preparing classifier inputs...

Metadata columns verified.
Number of feature columns found: 26
Feature count verified.
No missing values detected.
Observed class labels: ['ai', 'rl']
Class labels verified.
Classifier input arrays prepared successfully.

X_train shape: (14400, 26)
y_train shape: (14400,)

Encoded label mapping:
  ai -> 0
  rl -> 1


### 🔷 Step 4 — Define Selected Classifier Configurations

- Define final classifier configurations carried forward from previous selection and tuning
- Include RBF SVM and MLP as the retained models
- Specify fully configured hyperparameters for each classifier
- Ensure configurations are ready for cross-validation
- Optionally display classifier configurations when `VERBOSE=True`

---

In [4]:
# ============================================
# Step 4: Define Selected Classifier Configurations
# ============================================

print("Defining selected classifier configurations...\n")

# -------------------------------------------------
# Define retained classifier configurations
# (based on prior selection and tuning results)
# -------------------------------------------------
classifier_configs = [
    {
        "classifier": "RBF SVM",
        "params": {
            "C": 100,
            "gamma": 0.01,
            "kernel": "rbf",
            "probability": True,
            "random_state": RANDOM_SEED
        }
    },
    {
        "classifier": "MLP",
        "params": {
            "hidden_layer_sizes": (64, 32),
            "alpha": 0.0001,
            "learning_rate_init": 0.001,
            "batch_size": 32,
            "max_iter": 500,
            "random_state": RANDOM_SEED
        }
    }
]

# -------------------------------------------------
# Optional verbose display of classifier configs
# -------------------------------------------------
if VERBOSE:
    print(f"Number of classifiers configured: {len(classifier_configs)}\n")

    for i, config in enumerate(classifier_configs, start=1):
        print(f"{i}. {config['classifier']}")
        print("   Parameters:")
        for param_name, param_value in config["params"].items():
            print(f"     {param_name} = {param_value}")
        print()



Defining selected classifier configurations...

Number of classifiers configured: 2

1. RBF SVM
   Parameters:
     C = 100
     gamma = 0.01
     kernel = rbf
     probability = True
     random_state = 42

2. MLP
   Parameters:
     hidden_layer_sizes = (64, 32)
     alpha = 0.0001
     learning_rate_init = 0.001
     batch_size = 32
     max_iter = 500
     random_state = 42



### 🔷 Step 5 — Run Stratified Cross-Validation

- Define stratified cross-validation strategy using configured parameters
- Define evaluation metrics (accuracy, precision, recall, F1, ROC AUC)
- Instantiate each selected classifier (RBF SVM and MLP)
- Perform cross-validation using the full training dataset
- Compute mean and standard deviation for each metric
- Store results for later comparison and reporting
- Optionally display performance metrics during execution when `VERBOSE=True`
- Confirm completion of cross-validation process

---

In [5]:
# ============================================
# Step 5: Run Stratified Cross-Validation for Both Classifiers
# ============================================

print("Running stratified cross-validation for selected classifiers...\n")

# -------------------------------------------------
# Define cross-validation strategy
# -------------------------------------------------
cv = StratifiedKFold(
    n_splits=K_FOLDS,
    shuffle=CV_SHUFFLE,
    random_state=CV_RANDOM_STATE
)

# -------------------------------------------------
# Define evaluation metrics
# -------------------------------------------------
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# -------------------------------------------------
# Initialize storage for cross-validation results
# -------------------------------------------------
cv_results_list = []

# -------------------------------------------------
# Evaluate each configured classifier
# -------------------------------------------------
for config in classifier_configs:
    classifier_name = config["classifier"]
    params = config["params"]

    print(f"Evaluating: {classifier_name}")

    # -------------------------------------------------
    # Instantiate model
    # -------------------------------------------------
    if classifier_name == "RBF SVM":
        model = SVC(**params)
    elif classifier_name == "MLP":
        model = MLPClassifier(**params)
    else:
        raise ValueError(f"Unsupported classifier: {classifier_name}")

    # -------------------------------------------------
    # Perform cross-validation
    # -------------------------------------------------
    scores = cross_validate(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    # -------------------------------------------------
    # Aggregate results
    # -------------------------------------------------
    result_row = {
        "classifier": classifier_name,
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "precision_mean": np.mean(scores["test_precision"]),
        "precision_std": np.std(scores["test_precision"]),
        "recall_mean": np.mean(scores["test_recall"]),
        "recall_std": np.std(scores["test_recall"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }

    cv_results_list.append(result_row)

    # -------------------------------------------------
    # Optional verbose output
    # -------------------------------------------------
    if VERBOSE:
        print(f"Completed: {classifier_name}")
        print(f"  Mean ROC-AUC:  {result_row['roc_auc_mean']:.4f}")
        print(f"  Mean F1-score: {result_row['f1_mean']:.4f}\n")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("Cross-validation complete.")



Running stratified cross-validation for selected classifiers...

Evaluating: RBF SVM
Completed: RBF SVM
  Mean ROC-AUC:  0.8794
  Mean F1-score: 0.8007

Evaluating: MLP
Completed: MLP
  Mean ROC-AUC:  0.8403
  Mean F1-score: 0.7650

Cross-validation complete.


### 🔷 Step 6 — Summarize and Compare Validation Performance

- Convert cross-validation results into a structured DataFrame
- Verify that validation results are available
- Sort classifiers by ROC AUC (primary evaluation metric)
- Create a condensed comparison table of key metrics:
  - Accuracy
  - Precision
  - Recall
  - F1 Score
  - ROC AUC
- Optionally display full and condensed results when `VERBOSE=True`
- Prepare results for saving and final comparison

---

In [6]:
# ============================================
# Step 6: Summarize and Compare Validation Performance
# ============================================

print("Summarizing validation performance...\n")

# -------------------------------------------------
# Convert cross-validation results to DataFrame
# -------------------------------------------------
df_cv_results = pd.DataFrame(cv_results_list)

# -------------------------------------------------
# Validate results are available
# -------------------------------------------------
if df_cv_results.empty:
    raise ValueError("No cross-validation results found.")

# -------------------------------------------------
# Sort results by primary metric (ROC-AUC)
# -------------------------------------------------
df_cv_results = df_cv_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------
# Create condensed comparison table
# -------------------------------------------------
df_tuned_comparison = df_cv_results[
    [
        "classifier",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "roc_auc_mean"
    ]
].copy()

print("Validation results summarized successfully.")

# -------------------------------------------------
# Optional verbose display of results
# -------------------------------------------------
if VERBOSE:
    print("\nValidation results ranked by ROC-AUC:\n")
    try:
        display(df_cv_results)
    except:
        print(df_cv_results)

    print("\nCondensed tuned-model comparison:\n")
    try:
        display(df_tuned_comparison)
    except:
        print(df_tuned_comparison)



Summarizing validation performance...

Validation results summarized successfully.

Validation results ranked by ROC-AUC:



,classifier,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,RBF SVM,0.798889,0.008192,0.793584,0.010080,0.808056,0.005886,0.800738,0.007493,0.879407,0.005356
1,MLP,0.757639,0.006161,0.742920,0.015241,0.789306,0.019610,0.765044,0.004912,0.840305,0.004546



Condensed tuned-model comparison:



,classifier,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean
0,RBF SVM,0.798889,0.793584,0.808056,0.800738,0.879407
1,MLP,0.757639,0.742920,0.789306,0.765044,0.840305


### 🔷 Step 7 — Save Validation Outputs

- Save full cross-validation results to CSV file
- Save condensed tuned-model comparison table to CSV
- Ensure outputs are stored in the configured results directory
- Optionally display saved file paths when `VERBOSE=True`
- Confirm successful persistence of validation outputs

---

In [7]:
# ============================================
# Step 7: Save Validation Outputs
# ============================================

print("Saving validation outputs...\n")

# -------------------------------------------------
# Save full cross-validation results
# -------------------------------------------------
df_cv_results.to_csv(CV_RESULTS_CSV_PATH, index=False)

if VERBOSE:
    print(f"Saved cross-validation results: {CV_RESULTS_CSV_PATH}")

# -------------------------------------------------
# Save condensed tuned-model comparison table
# -------------------------------------------------
df_tuned_comparison.to_csv(TUNED_COMPARISON_CSV_PATH, index=False)

if VERBOSE:
    print(f"Saved tuned comparison results: {TUNED_COMPARISON_CSV_PATH}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("\nValidation outputs saved successfully.")



Saving validation outputs...

Saved cross-validation results: /content/dip-ai-image-detection/metadata/results/cross_validation_tuned_results.csv
Saved tuned comparison results: /content/dip-ai-image-detection/metadata/results/classifier_comparison_tuned.csv

Validation outputs saved successfully.


### 🔷 Step 8 — Validation Output Sanity Check

- Reload saved cross-validation and comparison CSV files
- Verify output file integrity and structure
- Optionally display formatted previews of validation results when `VERBOSE=True`
- Present side-by-side comparison of tuned classifier performance
- Confirm that expected classifiers (RBF SVM and MLP) are present
- Validate consistency between saved outputs
- Confirm successful completion of validation workflow

---

In [8]:
# ============================================
# Step 8: Validation Output Sanity Check
# ============================================

print("Performing sanity check on saved validation outputs...\n")

# -------------------------------------------------
# Reload saved validation output files
# -------------------------------------------------
df_cv_check = pd.read_csv(CV_RESULTS_CSV_PATH)
df_tuned_check = pd.read_csv(TUNED_COMPARISON_CSV_PATH)

# -------------------------------------------------
# Optional verbose display of outputs
# -------------------------------------------------
if VERBOSE:
    print("Cross-validation results shape:", df_cv_check.shape)
    print("Tuned comparison shape:", df_tuned_check.shape)

    print("\nCross-validation results preview:")
    try:
        display(
            df_cv_check.head().style
            .format("{:.4f}", subset=df_cv_check.select_dtypes(include="number").columns)
            .highlight_max(axis=0, color="lightgreen")
        )
    except:
        print(df_cv_check.head())

    print("\nSide-by-side tuned-model comparison:")

    comparison_check = df_tuned_check.set_index("classifier").T

    try:
        display(
            comparison_check.style
            .format("{:.4f}")
            .highlight_max(axis=1, color="lightgreen")
            .set_properties(**{"text-align": "center"})
        )
    except:
        print(comparison_check)

# -------------------------------------------------
# Validate expected classifiers are present
# -------------------------------------------------
expected_classifiers = {"RBF SVM", "MLP"}

cv_classifiers = set(df_cv_check["classifier"].unique())
tuned_classifiers = set(df_tuned_check["classifier"].unique())

if cv_classifiers != expected_classifiers:
    raise ValueError(f"Unexpected classifiers in CV results: {cv_classifiers}")

if tuned_classifiers != expected_classifiers:
    raise ValueError(f"Unexpected classifiers in tuned comparison: {tuned_classifiers}")

# -------------------------------------------------
# Final confirmation
# -------------------------------------------------
print("Sanity check passed: expected classifiers present in both outputs.")
print("Validation notebook completed successfully.")



Performing sanity check on saved validation outputs...

Cross-validation results shape: (2, 11)
Tuned comparison shape: (2, 6)

Cross-validation results preview:


,classifier,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,RBF SVM,0.7989,0.0082,0.7936,0.0101,0.8081,0.0059,0.8007,0.0075,0.8794,0.0054
1,MLP,0.7576,0.0062,0.7429,0.0152,0.7893,0.0196,0.7650,0.0049,0.8403,0.0045



Side-by-side tuned-model comparison:


classifier,RBF SVM,MLP
accuracy_mean,0.7989,0.7576
precision_mean,0.7936,0.7429
recall_mean,0.8081,0.7893
f1_mean,0.8007,0.7650
roc_auc_mean,0.8794,0.8403


Sanity check passed: expected classifiers present in both outputs.
Validation notebook completed successfully.
